## Exercise 6 Markov chains

In [ ]:
# task 1 The number of busy lines in a trunk group (Erlang system) is
#given by a truncated Poisson distribution 
# P(i) = c A^i/i!    , i=0,...,m

# generate values form this dsitribution by appluing the metropolis-Hastings 
#algorithm, verify with a X^2 -test. You can use the aprameter values form 
# exercise 4 
import numpy as np
import math
import scipy.stats as stats
np.random.seed(42)

N_CUSTUMERS = 10_000
burn_in = 100
M_SERVERS = 10
MU_SERVICE_TIME = 8
MU_ARRIVAL_TIME = 1
lam = 1
N_REPS = 10

A = MU_SERVICE_TIME / MU_ARRIVAL_TIME


def target(i, A):
    return A**i / math.factorial(i)

def q_prob(x, y, m):
    if x == 0:
        return 0.5 if y == 0 else 0.5 if y == 1 else 0
    elif x == m:
        return 0.5 if y == m else 0.5 if y == m-1 else 0
    else:
        return 0.5 if abs(y-x) == 1 else 0
    

weights = np.array([target(i, A) for i in range(M_SERVERS + 1)])
probs = weights / np.sum(weights)
for i,p in enumerate(probs):
    print(i,p)
def metropolis_hastings(A, m_servers, N, start=0):
    samples = np.zeros(N, dtype=int)
    current = start

    for t in range(N):
        # propose moving -1 or +1
        proposal = current + np.random.choice([-1, 1])

        # reject immediately if outside allowed range

        if proposal < 0 or proposal > m_servers:
            samples[t] = current
            continue

        # acceptance probability
        alpha = min(
            1,
            (target(proposal, A) * q_prob(proposal, current, m_servers)) /
            (target(current, A) * q_prob(current, proposal, m_servers))
        )

        if np.random.rand() < alpha:
            current = proposal

        samples[t] = current

    return samples

0 0.00041116370815887153
1 0.0032893096652709722
2 0.013157238661083889
3 0.03508596976289037
4 0.07017193952578074
5 0.11227510324124919
6 0.14970013765499893
7 0.17108587160571304
8 0.17108587160571304
9 0.1520763303161894
10 0.12166106425295149


In [39]:
np.random.seed(42)

samples = metropolis_hastings(A, M_SERVERS, N_CUSTUMERS, start=0)
steady_state_samples = samples[burn_in::10]

observed_counts = np.bincount(steady_state_samples, minlength=M_SERVERS+1)
observed_probs = observed_counts / len(steady_state_samples)
expected_counts = len(steady_state_samples) * probs

print("Observed probabilities:")
for i, p in enumerate(observed_probs):
    print(f"P_hat({i}) = {p:.4f}")


# mean, lower, upper = compute_ci(steady_state_samples)
# print(f"mean: {np.round(mean,4)}, Confidence 95%: [{np.round(lower,4)}, {np.round(upper,4)}] ")

expected_counts = len(steady_state_samples) * probs

chi2_stat, p_value = stats.chisquare(
    f_obs=observed_counts,
    f_exp=expected_counts
)

print("Chi-square statistic:", chi2_stat)
print("p-value:", p_value)
print(f"Observed counts: {observed_counts}")
print(f"Expectetd: {np.round(expected_counts,1)}")

if p_value < 0.05:
    print("Reject H0: samples do not fit the target distribution.")
else:
    print("Do not reject H0: samples fit the target distribution.")

Observed probabilities:
P_hat(0) = 0.0000
P_hat(1) = 0.0051
P_hat(2) = 0.0121
P_hat(3) = 0.0343
P_hat(4) = 0.0657
P_hat(5) = 0.1212
P_hat(6) = 0.1374
P_hat(7) = 0.1879
P_hat(8) = 0.1636
P_hat(9) = 0.1525
P_hat(10) = 0.1202
Chi-square statistic: 5.405251373076491
p-value: 0.8625168724860388
Observed counts: [  0   5  12  34  65 120 136 186 162 151 119]
Expectetd: [  0.4   3.3  13.   34.7  69.5 111.2 148.2 169.4 169.4 150.6 120.4]
Do not reject H0: samples fit the target distribution.


_____

In [5]:
# part 2 For the two different call types the joint number of the occupied lines is given by 
# # P(i,j) = c A^i_1/i! * A^j_2 /j!  , o≤i+j ≤m 

# 2 (a)
import numpy as np
import math
import scipy.stats as stats
M_SERVERS = 10
N_CUSTUMERS = 10_000

np.random.seed(42)
A1, A2 = 4, 4
m = M_SERVERS
thin = 1
burn_in = 1000

def target(state, A1, A2):
    i, j = state
    return (A1**i / math.factorial(i)) * (A2**j / math.factorial(j))

def metropolis_hastings_direct(A1, A2, m, N, start=(0,0)):
    samples = np.zeros((N, 2), dtype=int)

    states = []
    for i in range(m + 1):
        for j in range(m + 1):
            if i + j <= m:
                states.append((i, j))

    current = start

    for t in range(N):
        proposal = states[np.random.randint(len(states))]

        alpha = min(1, target(proposal, A1, A2) / target(current, A1, A2))

        if np.random.rand() < alpha:
            current = proposal

        samples[t] = current

    return samples


# testing 


samples = metropolis_hastings_direct(A1, A2, m, N_CUSTUMERS, start=(0,0))


steady_state_samples = samples[burn_in:]

states = []
weights = []

for i in range(m + 1):
    for j in range(m + 1):
        if i + j <= m:
            states.append((i, j))
            weights.append(target((i, j), A1, A2))

weights = np.array(weights)
probs = weights / weights.sum()

observed_counts = []

for i, j in states:
    count = np.sum(
        (steady_state_samples[:, 0] == i) &
        (steady_state_samples[:, 1] == j)
    )
    observed_counts.append(count)

observed_counts = np.array(observed_counts)
expected_counts = len(steady_state_samples) * probs

mask = expected_counts >= 5

obs_test = np.append(observed_counts[mask], observed_counts[~mask].sum())
exp_test = np.append(expected_counts[mask], expected_counts[~mask].sum())

chi2_stat, p_value = stats.chisquare(
    f_obs=obs_test,
    f_exp=exp_test
)

print("Chi-square statistic:", chi2_stat)
print("p-value:", p_value)

if p_value < 0.05:
    print("Reject H0")
else:
    print("Do not reject H0")

Chi-square statistic: 163.93062331102672
p-value: 2.2966324167165667e-11
Reject H0


_____

In [ ]:

np.random.seed(42)
A1, A2 = 4, 4
m = M_SERVERS
thin = 1
burn_in = 1000

# (b) Use metropolis- Hastings coordinate wise

def target(state, A1, A2):
    i, j = state
    return (A1**i / math.factorial(i)) * (A2**j / math.factorial(j))

def q_prob(x, y, m):
    if x == 0:
        return 0.5 if y == 0 else 0.5 if y == 1 else 0
    elif x == m:
        return 0.5 if y == m else 0.5 if y == m-1 else 0
    else:
        return 0.5 if abs(y-x) == 1 else 0
    




def valid_neighbors(state, m):
    i, j = state
    possible = [(i+1,j), (i-1,j), (i,j+1), (i,j-1)]

    return [
        (a,b) for a,b in possible
        if a >= 0 and b >= 0 and a + b <= m
    ]


def metropolis_hastings(A1, A2, m, N, start=(0,0)):
    samples = np.zeros((N, 2), dtype=int)

    states = []
    for i in range(m + 1):
        for j in range(m + 1):
            if i + j <= m:
                states.append((i, j))

    current = start

    for t in range(N):
        proposal = states[np.random.randint(len(states))]

        alpha = min(
            1,
            target(proposal, A1, A2) / target(current, A1, A2)
        )

        if np.random.rand() < alpha:
            current = proposal

        samples[t] = current

    return samples


np.random.seed(42)

samples = metropolis_hastings(A1, A2, M_SERVERS, N_CUSTUMERS, start=(0,0))
steady_state_samples = samples[burn_in::thin]

states = []
weights = []

for i in range(m + 1):
    for j in range(m + 1):
        if i + j <= m:
            states.append((i, j))
            weights.append(target((i, j), A1, A2))

weights = np.array(weights)
probs = weights / weights.sum()


observed_counts = []

for state in states:
    i, j = state
    count = np.sum(
        (steady_state_samples[:, 0] == i) &
        (steady_state_samples[:, 1] == j)
    )
    observed_counts.append(count)

observed_counts = np.array(observed_counts)
observed_probs = observed_counts / len(steady_state_samples)


expected_counts = len(steady_state_samples) * probs


chi2_stat, p_value = stats.chisquare(
    f_obs=observed_counts,
    f_exp=expected_counts
)

# print("Chi-square statistic:", chi2_stat)
# print("p-value:", p_value)

print(f"Observed: {observed_counts}")
print(f"Expected: {np.round(expected_counts,1)}")
print(f"excpected_counts<5= {np.sum(expected_counts < 5)}")

# if p_value < 0.05:
#     print("Reject H0")
# else:
#     print("Do not reject H0")

mask = expected_counts >= 5

obs_test = np.append(observed_counts[mask], observed_counts[~mask].sum())
exp_test = np.append(expected_counts[mask], expected_counts[~mask].sum())

chi2_stat, p_value = stats.chisquare(obs_test, exp_test)

print(chi2_stat, p_value)

accept_rate = np.mean(np.any(samples[1:] != samples[:-1], axis=1))
print("Acceptance/move rate:", accept_rate)